# 01. LangChain 핵심 복습

## 학습 목표
- LCEL(LangChain Expression Language) 문법을 복습하고 `prompt | model | parser` 체인을 구성할 수 있다
- `RunnablePassthrough`, `RunnableLambda`를 활용한 유연한 체인을 만들 수 있다
- ChromaDB를 활용한 RAG 파이프라인을 구축할 수 있다
- `@tool` 데코레이터로 Tool을 정의하고 LLM에 바인딩하여 수동 실행 루프를 구현할 수 있다

## 1. 환경 설정

In [2]:
# 필요한 패키지 설치 (이미 설치된 경우 스킵)
# !pip install langchain langchain-openai langchain-community langchain-core chromadb python-dotenv jupyter ipykernel langgraph

from dotenv import load_dotenv
load_dotenv()

# 환경 변수 확인
import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env 파일에 설정되어 있는지 확인하세요!"
print("환경 설정 완료!")

환경 설정 완료!


## 2. LCEL 기본: prompt | model | parser 체인

LCEL은 LangChain의 핵심 문법입니다. 파이프(`|`) 연산자로 컴포넌트를 연결합니다.

```
prompt → model → parser
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 생성 (gpt-5-mini는 temperature 파라미터를 지원하지 않습니다)
llm = ChatOpenAI(model="gpt-5-mini")

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다. 간결하게 답변하세요."),
    ("human", "{question}")
])

# 출력 파서
parser = StrOutputParser()

# LCEL 체인 구성: prompt | model | parser
chain = prompt | llm | parser

# 실행
result = chain.invoke({
    "role": "파이썬 프로그래밍",
    "question": "리스트 컴프리헨션이 뭔가요?"
})
print(result)

## 3. RunnablePassthrough, RunnableLambda 활용

- `RunnablePassthrough`: 입력을 그대로 다음 단계로 전달
- `RunnableLambda`: 임의의 Python 함수를 Runnable로 변환

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# RunnablePassthrough 예제: 입력을 그대로 전달하면서 추가 키 삽입
chain_with_passthrough = (
    {
        "question": RunnablePassthrough(),  # 입력 문자열을 그대로 question에 전달
        "role": lambda x: "AI"  # 고정값 삽입
    }
    | prompt
    | llm
    | parser
)

result = chain_with_passthrough.invoke("트랜스포머 아키텍처를 설명해주세요")
print("=== RunnablePassthrough 결과 ===")
print(result)
print()

# RunnableLambda 예제: 커스텀 전처리 함수
def preprocess(input_text: str) -> dict:
    """입력 텍스트를 전처리하여 dict로 변환"""
    return {
        "role": "데이터 사이언스",
        "question": f"초보자에게 쉽게 설명해주세요: {input_text}"
    }

chain_with_lambda = (
    RunnableLambda(preprocess)
    | prompt
    | llm
    | parser
)

result = chain_with_lambda.invoke("경사하강법")
print("=== RunnableLambda 결과 ===")
print(result)

## 4. RAG 파이프라인 (ChromaDB + OpenAIEmbeddings + Retriever)

RAG(Retrieval-Augmented Generation): 외부 문서를 검색하여 LLM 응답에 활용하는 패턴

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma

# 1) 샘플 문서 준비
documents = [
    "LangChain은 LLM 기반 애플리케이션을 쉽게 개발할 수 있게 해주는 프레임워크입니다.",
    "LangGraph는 LangChain 위에 구축된 에이전트 오케스트레이션 프레임워크로, 상태 기반 그래프를 사용합니다.",
    "RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식을 검색하여 LLM 응답을 보강하는 기법입니다.",
    "ChromaDB는 벡터 데이터베이스로, 임베딩 벡터를 저장하고 유사도 검색을 수행합니다.",
    "LCEL은 LangChain Expression Language로, 파이프 연산자를 사용하여 체인을 구성합니다.",
]

# 2) 임베딩 모델 & ChromaDB 벡터 스토어 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(
    texts=documents,
    embedding=embeddings,
    collection_name="langchain_review"
)

# 3) Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 4) RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 컨텍스트를 기반으로 질문에 답변하세요.\n\n컨텍스트:\n{context}"),
    ("human", "{question}")
])

# 5) 검색 결과를 문자열로 변환하는 함수
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

# 6) RAG 체인 구성
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser
)

# 7) 실행
result = rag_chain.invoke("LangGraph가 뭔가요?")
print(result)

## 5. Tool 정의 + bind_tools + 수동 실행 루프

LLM이 도구를 "호출하겠다"고 결정하면, 우리가 직접 실행하고 결과를 돌려주는 패턴입니다.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# Tool 정의
@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    # 실제로는 API를 호출하겠지만, 여기서는 시뮬레이션
    weather_data = {
        "서울": "맑음, 22도",
        "부산": "흐림, 19도",
        "제주": "비, 18도"
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

# Tool 목록
tools = [add, multiply, get_weather]
tool_map = {t.name: t for t in tools}

# LLM에 tools 바인딩
llm_with_tools = llm.bind_tools(tools)

# 수동 실행 루프
def run_tool_loop(user_input: str):
    """Tool 호출을 수동으로 처리하는 루프"""
    messages = [HumanMessage(content=user_input)]
    
    while True:
        # LLM 호출
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Tool 호출이 없으면 최종 응답 반환
        if not response.tool_calls:
            return response.content
        
        # Tool 호출 실행
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            print(f"  [Tool 호출] {tool_name}({tool_args})")
            
            # Tool 실행
            result = tool_map[tool_name].invoke(tool_args)
            print(f"  [Tool 결과] {result}")
            
            # 결과를 메시지에 추가
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            ))

# 테스트
print("=== 계산 요청 ===")
result = run_tool_loop("15와 27을 더하고, 그 결과에 3을 곱해주세요.")
print(f"최종 답변: {result}")
print()

print("=== 날씨 조회 ===")
result = run_tool_loop("서울과 부산의 날씨를 알려주세요.")
print(f"최종 답변: {result}")

## 6. 실습 과제: 자유 주제 RAG 체인 만들기

아래 조건을 만족하는 RAG 체인을 직접 만들어 보세요.

### 요구사항
1. 자유 주제의 문서 5개 이상을 준비하세요 (예: 좋아하는 영화, 맛집, 여행지 등)
2. ChromaDB에 저장하고 retriever를 생성하세요
3. RAG 체인을 구성하여 질문에 답변하는 시스템을 만드세요
4. 최소 3개의 질문으로 테스트하세요

In [ ]:
# 여기에 실습 코드를 작성하세요

# 1) 문서 준비
my_documents = [
    # 예시: "부산의 유명 맛집 해운대 XX식당은 해물탕이 유명합니다."
    # 여러분의 문서를 작성하세요!
]

# 2) ChromaDB에 저장


# 3) RAG 체인 구성


# 4) 테스트
